In [1]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re

from renewables_permitting.utils import as_list, save_parquet

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

In [ ]:
boe_items = pd.read_parquet(
    SILVER_DIR / "boe_items" / "boe_items_normalized.parquet"
)

In [3]:
# TODO: ajustar keywords y añadir más si es necesario

keywords_departamento = [
    "energia",
    "industria",
    "transicion ecologica",
    "movilidad",
    "medio ambiente",
    "politica territorial",
    "transportes"
]

keywords_epigrafe = [
    "energia electrica",
    "impacto ambiental",
    "instalaciones electricas",
    "otros anuncios oficiales"
]

keywords_titulo = [
    "almacenamiento",
    "autorizacion administrativa de construccion",
    "autorizacion administrativa previa",
    "bateria",
    "declaracion de impacto ambiental",
    "energia electrica",
    "energia solar",
    "eolic",
    "evacuacion",
    "fotovoltaic",
    "hibridacion",
    "impacto ambiental",
    "informacion publica",
    "infraestructura de evacuacion",
    "linea de evacuacion",
    "linea electrica",
    "parque eolico",
    "planta fotovoltaica",
    "planta solar",
    "plantas solares",
    "repotenciacion",
    "solar termica",
    "subestacion",
    "utilidad publica",
]

In [4]:
pattern_departamento = "|".join(
    re.escape(k) for k in keywords_departamento
)

pattern_epigrafe = "|".join(
    re.escape(k) for k in keywords_epigrafe
)

pattern_titulo = "|".join(
    re.escape(k) for k in keywords_titulo
)

In [5]:
mask_departamento = (
    boe_items["departamento_nombre_norm"]
    .str.contains(pattern_departamento, regex=True, na=False)
)

mask_epigrafe = (
    boe_items["epigrafe_nombre_norm"]
    .str.contains(pattern_epigrafe, regex=True, na=False)
)

mask_titulo = (
    boe_items["titulo_norm"]
    .str.contains(pattern_titulo, regex=True, na=False)
)

# mask =  mask_departamento | mask_epigrafe | mask_titulo
mask = mask_titulo

boe_candidates_normalized = boe_items.loc[mask].copy()

# boe_candidates_normalized.loc[boe_candidates_normalized["titulo_norm"].notnull(), "titulo_norm"].values[0:10]

In [6]:
save_parquet(
    boe_candidates_normalized,
    SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet",
)

# Pruebas

In [7]:
test = pd.read_parquet(
    SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"
)

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 980 entries, 0 to 979
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   identificador             980 non-null    object
 1   control                   106 non-null    object
 2   titulo                    980 non-null    object
 3   url_html                  980 non-null    object
 4   url_xml                   980 non-null    object
 5   url_pdf                   980 non-null    object
 6   pdf_size_bytes            980 non-null    object
 7   pdf_size_kbytes           980 non-null    object
 8   pagina_inicial            980 non-null    object
 9   pagina_final              980 non-null    object
 10  fecha_publicacion         980 non-null    object
 11  year                      980 non-null    int64 
 12  month                     980 non-null    int64 
 13  day                       980 non-null    int64 
 14  diario_numero             

In [ ]:
# test.loc[test["epigrafe_nombre"].isnull()]

In [12]:
test[1:2].values

array([['BOE-A-2023-10298', '2023/10000',
        'Resolución de 17 de abril de 2023, de la Dirección General de Política Energética y Minas, por la que se otorga a Varadero Solar, SLU, autorización administrativa previa para la instalación fotovoltaica Varadero Solar, de 47,71 MW de potencia instalada, y sus infraestructuras de evacuación, en Arganda del Rey y Loeches (Madrid).',
        'https://www.boe.es/diario_boe/txt.php?id=BOE-A-2023-10298',
        'https://www.boe.es/diario_boe/xml.php?id=BOE-A-2023-10298',
        'https://www.boe.es/boe/dias/2023/04/28/pdfs/BOE-A-2023-10298.pdf',
        '229972', '225', '59006', '59012', '2023-04-28', 2023, 4, 28,
        '101', '3', 'III. Otras disposiciones', '9575',
        'MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO DEMOGRÁFICO',
        'Instalaciones eléctricas', 'departamento.epigrafe.item', 'boe',
        'ES', '20230428', 'iii otras disposiciones',
        'ministerio para la transicion ecologica y el reto demografico',
    